In [1]:
# ================= MIXED-PRECISION QAT | CIFAR-10 | RESNET18 =================
!pip install torch torchvision timm tqdm --quiet

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm
import torch.ao.quantization as quant

# ---------------- Setup ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 100
WARMUP_EPOCHS = 30
BATCH_SIZE = 128
NUM_CLASSES = 10

# ---------------- CIFAR-10 ----------------
MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2470, 0.2435, 0.2616]

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = torchvision.datasets.CIFAR10(
    "./data", train=True, download=True, transform=train_tf
)
test_ds = torchvision.datasets.CIFAR10(
    "./data", train=False, download=True, transform=test_tf
)

train_ld = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2)
test_ld  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2)

# ---------------- Mixed-Precision ResNet18 ----------------
class MixedQATResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        base = timm.create_model(
            "resnet18", pretrained=True, num_classes=NUM_CLASSES
        )

        # CIFAR-friendly stem
        base.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        base.maxpool = nn.Identity()

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.act1 = base.act1

        self.quant = quant.QuantStub()
        self.dequant = quant.DeQuantStub()

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.pool = base.global_pool
        self.fc = base.fc

    def forward(self, x):
        x = self.act1(self.bn1(self.conv1(x)))

        x = self.quant(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.dequant(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

model = MixedQATResNet18().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4
)

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[60, 85], gamma=0.1
)

# ---------------- Evaluation ----------------
def evaluate(m):
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_ld:
            x, y = x.to(device), y.to(device)
            out = m(x)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

# ---------------- Training ----------------
best_acc = 0.0
qat_enabled = False

for epoch in range(EPOCHS):
    model.train()

    for x, y in tqdm(train_ld, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    scheduler.step()

    acc = evaluate(model)
    print(f"[Epoch {epoch+1}] Val Acc: {acc:.4f}")

    # -------- ENABLE QAT (FIXED) --------
    if epoch + 1 == WARMUP_EPOCHS and not qat_enabled:
        print(">>> Enabling QAT")
        model.train()  # 🔑 REQUIRED LINE
        model.qconfig = quant.get_default_qat_qconfig("fbgemm")
        quant.prepare_qat(model, inplace=True)

        for g in optimizer.param_groups:
            g["lr"] = 1e-4

        qat_enabled = True

    if acc > best_acc:
        best_acc = acc
        acc_tag = f"{best_acc*100:.2f}".replace(".", "_")
        torch.save(
            model.state_dict(),
            f"best_qat_resnet18_cifar10_acc_{acc_tag}.pth"
        )

# ---------------- Convert to REAL INT8 ----------------
model.cpu().eval()
int8_model = quant.convert(model, inplace=False)

acc_tag = f"{best_acc*100:.2f}".replace(".", "_")
torch.save(
    int8_model.state_dict(),
    f"qat_resnet18_cifar10_int8_acc_{acc_tag}.pth"
)

100%|██████████| 170M/170M [00:15<00:00, 11.1MB/s] 


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Epoch 1/100: 100%|██████████| 391/391 [00:39<00:00, 10.00it/s]


[Epoch 1] Val Acc: 0.6325


Epoch 2/100: 100%|██████████| 391/391 [00:40<00:00,  9.69it/s]


...

Epoch 99/100: 100%|██████████| 391/391 [00:50<00:00,  7.82it/s]


[Epoch 99] Val Acc: 0.9191


Epoch 100/100: 100%|██████████| 391/391 [00:50<00:00,  7.81it/s]


[Epoch 100] Val Acc: 0.9155


/tmp/ipykernel_55/3157614060.py:154: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  int8_model = quant.convert(model, inplace=False)
